In [0]:
from pyspark.sql import Row

# 1. Simulation des données brutes reçues des capteurs
raw_data = [
    Row(car_id="CAR_01", lap=1, speed_kmh=245.5, engine_temp=92.0),
    Row(car_id="CAR_01", lap=1, speed_kmh=245.5, engine_temp=92.0), # Doublon !
    Row(car_id="CAR_01", lap=2, speed_kmh=-15.0, engine_temp=95.5), # Bug capteur (vitesse négative)
    Row(car_id="CAR_02", lap=1, speed_kmh=251.0, engine_temp=94.0),
    Row(car_id="CAR_02", lap=2, speed_kmh=258.2, engine_temp=None), # Valeur manquante
]

df_bronze = spark.createDataFrame(raw_data)

# 2. Sauvegarde dans la table Bronze (format Delta par défaut sur Databricks)
df_bronze.write.format("delta").mode("overwrite").saveAsTable("telemetry_bronze")

In [0]:
display(spark.table("telemetry_bronze"))

car_id,lap,speed_kmh,engine_temp
CAR_01,1,245.5,92.0
CAR_01,1,245.5,92.0
CAR_01,2,-15.0,95.5
CAR_02,1,251.0,94.0
CAR_02,2,258.2,null


In [0]:
from pyspark.sql.functions import col

# 1. Lecture de la table Bronze
df_bronze = spark.table("telemetry_bronze")

# 2. Nettoyage (Couche Silver)
df_silver = (
    df_bronze
    .dropDuplicates()                        # Supprime la ligne en double de CAR_01
    .filter(col("speed_kmh") > 0)            # Élimine le bug capteur (vitesse négative)
    .dropna(subset=["engine_temp"])          # Retire la ligne où la température est manquante (None)
)

# 3. Sauvegarde dans la table Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("telemetry_silver")

# 4. Affichage du résultat propre !
display(df_silver)

car_id,lap,speed_kmh,engine_temp
CAR_01,1,245.5,92.0
CAR_02,1,251.0,94.0


In [0]:
from pyspark.sql.functions import avg, max, count, round

# 1. Lecture de la table Silver propre
df_silver = spark.table("telemetry_silver")

# 2. Agrégation métier par voiture (Couche Gold)
df_gold = (
    df_silver
    .groupBy("car_id")
    .agg(
        count("lap").alias("valid_laps"),
        round(avg("speed_kmh"), 1).alias("avg_speed_kmh"),
        max("engine_temp").alias("max_engine_temp")
    )
)

# 3. Sauvegarde dans la table Gold
df_gold.write.format("delta").mode("overwrite").saveAsTable("telemetry_gold")

# 4. Affichage final
display(df_gold)

car_id,valid_laps,avg_speed_kmh,max_engine_temp
CAR_01,1,245.5,92.0
CAR_02,1,251.0,94.0


In [0]:
import random
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import current_timestamp, lit

# Fixer la graine aléatoire pour des résultats reproductibles
random.seed(42)

cars = ["CAR_01", "CAR_02", "CAR_03", "CAR_04", "CAR_05"]
raw_records = []

for car in cars:
    base_speed = random.uniform(242.0, 255.0)
    for lap in range(1, 101):  # 100 tours par voiture
        # Simulation normale
        speed = round(base_speed + random.uniform(-8.0, 12.0), 2)
        engine_temp = round(random.uniform(88.0, 104.0), 1)
        tire_pressure = round(random.uniform(21.5, 24.5), 2)
        lap_time = round(85.0 - (speed - 240.0) * 0.15 + random.uniform(-0.5, 0.5), 3)

        # Injection volontaire d'anomalies capteurs (~10% des données)
        anomaly_roll = random.random()
        if anomaly_roll < 0.03:
            speed = -99.0  # Bug capteur vitesse négative
        elif anomaly_roll < 0.06:
            engine_temp = None  # Sonde température déconnectée (NULL)
        elif anomaly_roll < 0.08:
            tire_pressure = 0.0  # Crevaison ou bug capteur pression

        record = (car, lap, speed, engine_temp, tire_pressure, lap_time)
        raw_records.append(record)

        # Injection volontaire de doublons réseau (~3% des lignes)
        if random.random() < 0.03:
            raw_records.append(record)

# Définition explicite du schéma (Bonne pratique Senior !)
schema = StructType([
    StructField("car_id", StringType(), True),
    StructField("lap", IntegerType(), True),
    StructField("speed_kmh", DoubleType(), True),
    StructField("engine_temp", DoubleType(), True),
    StructField("tire_pressure_psi", DoubleType(), True),
    StructField("lap_time_sec", DoubleType(), True)
])

# Création du DataFrame et ajout des colonnes d'audit
df_raw = spark.createDataFrame(raw_records, schema=schema)
df_bronze = (
    df_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_system", lit("telemetry_iot_gateway_v2"))
)

# Sauvegarde dans la table Bronze
df_bronze.write.format("delta").mode("overwrite").saveAsTable("racing_bronze")

print(f"Nombre total de lignes ingérées dans Bronze : {df_bronze.count()}")
display(df_bronze.limit(10))

Nombre total de lignes ingérées dans Bronze : 511


car_id,lap,speed_kmh,engine_temp,tire_pressure_psi,lap_time_sec,_ingested_at,_source_system
CAR_01,1,242.81,92.4,22.17,84.815,2026-11-18T08:04:15.805Z,telemetry_iot_gateway_v2
CAR_01,2,244.05,94.8,21.59,84.111,2026-11-18T08:04:15.805Z,telemetry_iot_gateway_v2
CAR_01,2,244.05,94.8,21.59,84.111,2026-11-18T08:04:15.805Z,telemetry_iot_gateway_v2
CAR_01,3,246.29,98.4,23.13,83.777,2026-11-18T08:04:15.805Z,telemetry_iot_gateway_v2
CAR_01,4,242.44,100.9,23.59,84.474,2026-11-18T08:04:15.805Z,telemetry_iot_gateway_v2
CAR_01,5,249.04,89.5,21.79,83.991,2026-11-18T08:04:15.805Z,telemetry_iot_gateway_v2
CAR_01,6,256.91,96.6,24.42,82.342,2026-11-18T08:04:15.805Z,telemetry_iot_gateway_v2
CAR_01,7,254.68,null,23.23,83.003,2026-11-18T08:04:15.805Z,telemetry_iot_gateway_v2
CAR_01,8,248.1,89.3,22.2,83.386,2026-11-18T08:04:15.805Z,telemetry_iot_gateway_v2
CAR_01,9,249.61,93.9,22.13,83.325,2026-11-18T08:04:15.805Z,telemetry_iot_gateway_v2


In [0]:
from pyspark.sql.functions import col, current_timestamp
from delta.tables import DeltaTable

# 1. Lecture de la table Bronze
df_bronze = spark.table("racing_bronze")

# 2. Dédoublonnage ciblé sur la clé métier (voiture + tour)
df_dedup = df_bronze.dropDuplicates(["car_id", "lap"])

# 3. Définition de la condition de qualité stricte
valid_condition = (
    (col("speed_kmh") > 0) & (col("speed_kmh") < 420) &
    (col("engine_temp").isNotNull()) & (col("engine_temp").between(50, 130)) &
    (col("tire_pressure_psi") > 15.0)
)

# 4. Séparation : Données valides -> Silver / Données rejetées -> Quarantaine
df_valid = df_dedup.filter(valid_condition).withColumn("_cleaned_at", current_timestamp())
df_quarantine = df_dedup.filter(~valid_condition).withColumn("_rejected_at", current_timestamp())

# Sauvegarde de la quarantaine
df_quarantine.write.format("delta").mode("overwrite").saveAsTable("racing_quarantine")

# 5. Chargement incrémental (MERGE / UPSERT) dans racing_silver
if not spark.catalog.tableExists("racing_silver"):
    # Première exécution : création de la table Silver
    df_valid.write.format("delta").saveAsTable("racing_silver")
else:
    # Exécutions suivantes : mise à jour incrémentale (Upsert)
    delta_silver = DeltaTable.forName(spark, "racing_silver")
    (
        delta_silver.alias("target")
        .merge(
            df_valid.alias("source"),
            "target.car_id = source.car_id AND target.lap = source.lap"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

print(f"Lignes valides dans Silver : {spark.table('racing_silver').count()}")
print(f"Lignes isolées en Quarantaine : {spark.table('racing_quarantine').count()}")
display(spark.table("racing_quarantine").limit(5))

Lignes valides dans Silver : 458
Lignes isolées en Quarantaine : 42


car_id,lap,speed_kmh,engine_temp,tire_pressure_psi,lap_time_sec,_ingested_at,_source_system,_rejected_at
CAR_01,15,246.7,null,23.03,83.586,2026-11-18T08:04:06.372Z,telemetry_iot_gateway_v2,2026-11-18T08:12:21.589Z
CAR_01,22,260.89,null,23.99,81.674,2026-11-18T08:04:06.372Z,telemetry_iot_gateway_v2,2026-11-18T08:12:21.589Z
CAR_01,33,260.78,null,24.08,81.933,2026-11-18T08:04:06.372Z,telemetry_iot_gateway_v2,2026-11-18T08:12:21.589Z
CAR_04,63,251.11,100.0,0.0,83.232,2026-11-18T08:04:06.372Z,telemetry_iot_gateway_v2,2026-11-18T08:12:21.589Z
CAR_01,45,-99.0,97.8,24.46,82.697,2026-11-18T08:04:06.372Z,telemetry_iot_gateway_v2,2026-11-18T08:12:21.589Z


In [0]:
from pyspark.sql.functions import avg, max, min, count, round, col

df_silver = spark.table("racing_silver")

df_gold = (
    df_silver
    .groupBy("car_id")
    .agg(
        count("lap").alias("valid_laps"),
        round(avg("speed_kmh"), 2).alias("avg_speed_kmh"),
        max("speed_kmh").alias("top_speed_kmh"),
        round(min("lap_time_sec"), 3).alias("best_lap_sec"),
        round(avg("lap_time_sec"), 3).alias("avg_lap_sec"),
        max("engine_temp").alias("max_engine_temp"),
        round(avg("tire_pressure_psi"), 2).alias("avg_tire_pressure")
    )
    .orderBy(col("best_lap_sec").asc()) # Classement du plus rapide au plus lent !
)

df_gold.write.format("delta").mode("overwrite").saveAsTable("racing_gold")

display(df_gold)

car_id,valid_laps,avg_speed_kmh,top_speed_kmh,best_lap_sec,avg_lap_sec,max_engine_temp,avg_tire_pressure
CAR_02,93,253.04,263.24,81.136,83.04,104.0,23.14
CAR_01,94,252.12,261.61,81.382,83.133,104.0,23.01
CAR_03,90,252.5,261.55,81.541,83.126,104.0,22.94
CAR_04,88,249.22,258.43,81.823,83.56,104.0,22.97
CAR_05,93,243.62,255.03,82.58,84.452,104.0,22.94


In [0]:
# 1. Simulation d'une erreur humaine : suppression d'une voiture en production
spark.sql("DELETE FROM racing_gold WHERE car_id = 'CAR_01'")
print("Après suppression accidentelle :")
display(spark.table("racing_gold"))

# 2. Consultation de l'historique transactionnel Delta Lake
display(spark.sql("DESCRIBE HISTORY racing_gold"))

# 3. Restauration instantanée à la version 0 (avant l'erreur !)
spark.sql("RESTORE TABLE racing_gold TO VERSION AS OF 0")
print("Après restauration Time Travel :")
display(spark.table("racing_gold"))

Après suppression accidentelle :


car_id,valid_laps,avg_speed_kmh,top_speed_kmh,best_lap_sec,avg_lap_sec,max_engine_temp,avg_tire_pressure
CAR_02,93,253.04,263.24,81.136,83.04,104.0,23.14
CAR_03,90,252.5,261.55,81.541,83.126,104.0,22.94
CAR_04,88,249.22,258.43,81.823,83.56,104.0,22.97
CAR_05,93,243.62,255.03,82.58,84.452,104.0,22.94


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-11-18T08:17:08.000Z,74769827543062,zeroxeazrl@gmail.com,DELETE,"Map(predicate -> [""(car_id#12629 = CAR_01)""])",null,List(1977670140738031),7d29ab19-e6ad-4f67-adc2-9672c4952c69,1118-080349-jcvqtsum-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3023, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 2214, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 782)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13
0,2026-11-18T08:16:48.000Z,74769827543062,zeroxeazrl@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1977670140738031),be9dffd0-bd62-4822-ab47-419498e833d5,1118-080349-jcvqtsum-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 2707)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13


Après restauration Time Travel :


car_id,valid_laps,avg_speed_kmh,top_speed_kmh,best_lap_sec,avg_lap_sec,max_engine_temp,avg_tire_pressure
CAR_02,93,253.04,263.24,81.136,83.04,104.0,23.14
CAR_01,94,252.12,261.61,81.382,83.133,104.0,23.01
CAR_03,90,252.5,261.55,81.541,83.126,104.0,22.94
CAR_04,88,249.22,258.43,81.823,83.56,104.0,22.97
CAR_05,93,243.62,255.03,82.58,84.452,104.0,22.94
